# 00c — pertpy Quickstart

**Datasets:** pertpy built-in examples (`norman_2019`, `papalexi_2021`)  
**Phase:** 1  
**Date:** 2026-03-11  
**Objective:** Walk through pertpy's core API — data loading, differential response, guide assignment, MIXSCAPE, and CINEMA-OT. Generous explanations are provided for every algorithm and statistical test.

## Table of Contents

1. [Setup](#setup)
2. [Data Loading](#data-loading)
   - 2a. `norman_2019` — combinatorial K562 CRISPRa screen
   - 2b. `papalexi_2021` — CITE-seq CRISPRi screen with guide RNA counts
3. [Differential Response](#diff-response) — identifying which genes change after perturbation
4. [Guide Assignment](#guide-assignment) — determining which guide RNA each cell received
5. [MIXSCAPE](#mixscape) — classifying cells as genuine KO vs. non-perturbed
6. [CINEMA-OT](#cinema-ot) — separating causal perturbation effects from confounding cell state
7. [Summary & phase checkpoint](#summary)

<a id='setup'></a>
## 0. Setup

In [ ]:
import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
import warnings
warnings.filterwarnings('ignore')

import pertpy as pt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False)
np.random.seed(42)

from pathlib import Path
RESULTS = Path('../../results')
FIG_DIR = RESULTS / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'pertpy version: {pt.__version__}')
print('Setup complete.')

<a id='data-loading'></a>
## 1. Data Loading — pertpy built-in datasets

pertpy ships several ready-to-use datasets that are fetched on demand and cached locally.
This is the fastest way to start exploring the API without needing to download or preprocess
your own data.

### 1a. `norman_2019` — combinatorial K562 CRISPRa screen

**Reference:** Norman, Weissman et al. (2019). *Exploring genetic interaction manifolds
constructed from rich single-cell phenotypes.* Science 365, 786–793.

**What it is:** A CRISPRa (activation) screen in K562 myeloid leukemia cells that includes:
- **Single perturbations:** one gene activated per cell
- **Double perturbations:** two genes co-activated per cell (combinatorial screen)
- **Control cells:** non-targeting guide RNA

**Why we use it here:**
- Moderate size (~105K cells, 287 perturbation groups) — fast enough for a tutorial
- Well-characterised TF perturbations with known biology (erythroid differentiation)
- Standard reference dataset for perturbation effect comparisons
- `nperts` column tells us whether a cell has 1 or 2 perturbations

### 1b. `papalexi_2021` — CITE-seq CRISPRi screen

**Reference:** Papalexi, Mimitou et al. (2021). *Characterizing the molecular regulation
of inhibitory immune checkpoints with multimodal single-cell screens.*
Nature Genetics 53, 322–331.

**What it is:** A CRISPRi (repression) screen combined with CITE-seq (simultaneous
mRNA + surface protein measurement) targeting immune checkpoint genes in immune cells.
It is the canonical dataset for demonstrating the **MIXSCAPE** algorithm.

**Why we use it here:**
- Has guide RNA count data (needed to demonstrate guide assignment)
- Real-world example of imperfect CRISPR efficiency (cells that received the guide
  but were not successfully knocked down)
- The perturbations target well-characterised immune checkpoint genes
  (PDCD1/PD-1, CD274/PD-L1, HAVCR2/TIM-3, etc.)
- `perturbation == 'NT'` labels non-targeting control cells

In [ ]:
# norman_2019 is returned as a standard AnnData object.
# pertpy downloads it on first use and caches to ~/.cache/pertpy/
adata = pt.data.norman_2019()
print(adata)
print()
print('obs columns:', adata.obs.columns.tolist())
print()
print('Perturbation type(s):', adata.obs['perturbation_type'].unique().tolist())
print('Cell line:', adata.obs['cell_line'].unique().tolist())
print()
pert_counts = adata.obs['perturbation'].value_counts()
print(f'Total cells:           {len(adata):,}')
print(f'Unique perturbations:  {adata.obs["perturbation"].nunique()}')
print(f'Control cells:         {(adata.obs["perturbation"] == "control").sum():,}')
print()
print('Top 12 perturbations by cell count:')
print(pert_counts.head(12))
print()
print('Cells with 1 perturbation:', (adata.obs['nperts'] == 1).sum())
print('Cells with 2 perturbations:', (adata.obs['nperts'] == 2).sum())

In [ ]:
# papalexi_2021 is returned as a MuData object (multi-modal AnnData).
# MuData stores multiple modalities, each as its own AnnData, under mdata.mod.
# The two modalities here are 'rna' (gene expression) and 'guide_RNA' (guide counts).
try:
    import mudata
    mdata = pt.data.papalexi_2021()
    print(type(mdata))
    print(mdata)
    print()
    adata_rna   = mdata['rna'].copy()
    adata_guide = mdata['guide_RNA'].copy()
except Exception as e:
    print(f'MuData load failed ({e}); trying AnnData fallback...')
    mdata = pt.data.papalexi_2021()
    adata_rna   = mdata.copy()
    adata_guide = None

print('RNA AnnData:', adata_rna.shape)
print('obs columns:', adata_rna.obs.columns.tolist())
print()
print('Perturbation labels (top 12):')
print(adata_rna.obs['perturbation'].value_counts().head(12))
print()
print('NT (non-targeting) cells:', (adata_rna.obs['perturbation'] == 'NT').sum())
if adata_guide is not None:
    print()
    print('Guide RNA AnnData:', adata_guide.shape)
    print('Guide names (first 8):', adata_guide.var_names[:8].tolist())

<a id='diff-response'></a>
## 2. Differential Response — which genes change after perturbation?

### What is differential response?

For each perturbation in a Perturb-seq screen, we want to know: **which genes are
up- or down-regulated compared to control cells?** This is the single-cell analogue
of differential expression analysis.

### Test options and their trade-offs

#### Wilcoxon rank-sum test (fast, but inflated FDR)
The simplest approach is to treat each cell as an independent observation and
compare expression ranks between the perturbed group and controls.

**How it works:**
1. Pool all cells from the perturbed group ($n$ cells) and the control group ($m$ cells).
2. Rank all $n + m$ cells by expression of gene $g$.
3. The Wilcoxon statistic $W$ = sum of ranks in the perturbed group.
4. Under the null (no effect), $W$ follows a known distribution → p-value.
5. Repeat for all genes; apply Benjamini-Hochberg FDR correction.

**The critical limitation:**
Cells within the same perturbation group are **not independent** — they were derived
from the same pool of cells under the same guide RNA delivery conditions and are
measured in the same sequencing batch. Treating them as independent violates the
test's assumptions and produces **inflated Type I error** (more false positives than
expected). A screen with 10 cells per perturbation will give the same p-value as
one with 1,000 cells, even though the latter has much more statistical power.

#### Pseudobulk DE with PyDESeq2 (recommended for final results)
The gold standard is to **aggregate counts within each perturbation group** ("pseudobulk")
and then run a count-based model (DESeq2 / PyDESeq2) that explicitly accounts for
population-level variation:

1. For each perturbation with cells from multiple guide RNA delivery events, sum raw
   counts across cells to create a "pseudobulk" sample per perturbation.
2. Run PyDESeq2: models counts with a negative binomial distribution, estimates
   dispersion from across-perturbation variation, and tests with a Wald test.
3. This approach requires **biological replicates** (multiple independent perturbation
   instances) to estimate variance — not always available in a single pooled screen.

**Rule of thumb:**
- Use Wilcoxon for *exploration* (it's fast; use as a first pass to rank perturbations)
- Use pseudobulk for *final publication results*

### What we show here
We use the Wilcoxon test via `sc.tl.rank_genes_groups` to quickly identify the
top DE genes for a well-characterised perturbation (CEBPE), then plot a volcano.

In [ ]:
# Restrict to single perturbations only (nperts == 1).
# Double perturbations are a separate analysis unit (combinatorial interaction screen).
adata_s = adata[adata.obs['nperts'] == 1].copy()
print(f'Single-perturbation cells: {len(adata_s):,}')

# Standard preprocessing pipeline: normalise → log → HVG → scale → PCA.
# We need a log-normalised representation for the Wilcoxon test (rank-based,
# but log-space reduces the influence of very high-expressing outlier cells).
sc.pp.normalize_total(adata_s, target_sum=1e4)
sc.pp.log1p(adata_s)

# Select 2000 HVGs: reduces noise from housekeeping genes not relevant to the screen.
sc.pp.highly_variable_genes(adata_s, n_top_genes=2000, subset=True)

print(f'After HVG selection: {adata_s.n_vars} genes')
print('Preprocessing complete.')

In [ ]:
# CEBPE is a C/EBP family transcription factor that activates erythroid differentiation
# in K562 cells — one of the strongest-effect perturbations in this dataset.
# We compare CEBPE-activated cells vs. non-targeting controls.
TARGET = 'CEBPE'
adata_de = adata_s[adata_s.obs['perturbation'].isin([TARGET, 'control'])].copy()
n_pert = (adata_de.obs['perturbation'] == TARGET).sum()
n_ctrl = (adata_de.obs['perturbation'] == 'control').sum()
print(f'{TARGET} cells: {n_pert},  control cells: {n_ctrl}')

# rank_genes_groups tests every gene in adata_de.var for differential expression.
# groupby='perturbation': the groups are defined by adata.obs['perturbation'].
# groups=[TARGET]: only test this specific group vs. reference.
# reference='control': the reference (denominator) group.
# method='wilcoxon': Wilcoxon rank-sum test (also supports 't-test', 'logreg').
# key_added: where to store results in adata.uns.
sc.tl.rank_genes_groups(
    adata_de,
    groupby='perturbation',
    groups=[TARGET],
    reference='control',
    method='wilcoxon',
    key_added='rank_genes_wilcoxon',
    pts=True,           # also compute fraction of cells expressing each gene
)

# Extract results as a DataFrame
result = sc.get.rank_genes_groups_df(
    adata_de,
    group=TARGET,
    key='rank_genes_wilcoxon',
)
print()
print(f'DE results: {len(result)} genes tested')
print()
print('Top 10 up-regulated genes:')
print(result.sort_values('scores', ascending=False).head(10).to_string(index=False))
print()
print('Top 10 down-regulated genes:')
print(result.sort_values('scores', ascending=True).head(10).to_string(index=False))

In [ ]:
# Volcano plot: log2 fold change (x) vs -log10 BH-adjusted p-value (y).
# Significant genes: |LFC| > 0.5 AND FDR-adjusted p < 0.05.
# The volcano shape arises because strongly regulated genes have both
# large fold changes AND small p-values.
df_vol = sc.get.rank_genes_groups_df(adata_de, group=TARGET, key='rank_genes_wilcoxon')
df_vol = df_vol.rename(columns={'logfoldchanges': 'lfc', 'pvals_adj': 'padj', 'names': 'gene'})
df_vol['padj'] = df_vol['padj'].clip(lower=1e-300)  # avoid log(0)
df_vol['neg_log10_padj'] = -np.log10(df_vol['padj'])

LFC_CUT = 0.5
FDR_CUT = 0.05
sig_mask = (df_vol['padj'] < FDR_CUT) & (df_vol['lfc'].abs() > LFC_CUT)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df_vol.loc[~sig_mask, 'lfc'], df_vol.loc[~sig_mask, 'neg_log10_padj'],
           s=10, alpha=0.4, color='steelblue', label='not significant', rasterized=True)
ax.scatter(df_vol.loc[sig_mask, 'lfc'], df_vol.loc[sig_mask, 'neg_log10_padj'],
           s=15, alpha=0.8, color='tomato', label=f'FDR<{FDR_CUT}, |LFC|>{LFC_CUT}', zorder=3)

# Label the top 8 hits by p-value
top8 = df_vol.nlargest(8, 'neg_log10_padj')
for _, row in top8.iterrows():
    ax.annotate(row['gene'], xy=(row['lfc'], row['neg_log10_padj']),
                xytext=(4, 2), textcoords='offset points', fontsize=7)

ax.axvline(LFC_CUT, color='gray', ls='--', lw=0.8)
ax.axvline(-LFC_CUT, color='gray', ls='--', lw=0.8)
ax.axhline(-np.log10(FDR_CUT), color='gray', ls='--', lw=0.8,
           label=f'FDR = {FDR_CUT}')
ax.set_xlabel('log2 fold change (perturbed / control)')
ax.set_ylabel(r'-log10(BH-adjusted p-value)')
ax.set_title(f'Differential response: {TARGET} CRISPRa vs. control\n'
             f'(Wilcoxon, n={n_pert} perturbed / n={n_ctrl} control cells, 2000 HVGs)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / f'00c_volcano_{TARGET}.pdf', bbox_inches='tight')
plt.show()

n_up   = ((sig_mask) & (df_vol['lfc'] > 0)).sum()
n_down = ((sig_mask) & (df_vol['lfc'] < 0)).sum()
print(f'Significant genes: {sig_mask.sum()} total  ({n_up} up, {n_down} down)')

### A note on pseudobulk DE (PyDESeq2)

For genome-scale screens (Phase 2 onwards), the preferred approach is **pseudobulk DE**:

```python
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

# 1. Aggregate raw counts per perturbation group
pb = sc.get.aggregate(adata_raw, by='perturbation', func='sum')   # raw counts
# 2. Create DESeq2 dataset
dds = DeseqDataSet(counts=pb.X, metadata=pb.obs, design_factors='perturbation')
dds.deseq2()
# 3. Test each perturbation vs. control
for pert in perturbations:
    stat_res = DeseqStats(dds, contrast=['perturbation', pert, 'control'])
    stat_res.summary()
```

This handles:
- **Overdispersion** in count data (negative binomial model)
- **Between-replicate variability** (critical for correct FDR control)
- **Library-size normalisation** internally via size factors

See `src/perturbation_analysis/differential.py` for the project wrapper.

<a id='guide-assignment'></a>
## 3. Guide Assignment — which guide RNA did each cell receive?

### The challenge

In a pooled Perturb-seq screen, a library of guide RNAs is delivered into a pool
of cells. After sequencing, you need to determine which guide RNA(s) each cell received.
This is not trivial:

- **Ambient guide RNA**: some cells will have low guide counts from ambient RNA in the
  cell suspension (not from a guide that was actually delivered)
- **Multiple guides**: some cells may have received more than one guide (doublets or
  multiplets in the library delivery)
- **Uninfected cells**: cells that received no guide (no perturbation, but no guide marker)

The guide RNA modality of the `papalexi_2021` dataset contains a UMI count matrix:
- Rows = cells
- Columns = guide RNA sequences
- Values = UMI counts (how many times that guide's sequence was detected in that cell)

### Assignment methods

#### 1. Threshold-based (`assign_by_threshold`)
A cell is considered 'positive' for a guide if its count exceeds a hard threshold.
Typical threshold: 5 UMIs. This is chosen to be above the ambient background
(1–3 UMIs from non-specific capture) but well below the signal from successful
guide delivery (typically 10–200 UMIs).

**Pros:** Simple, interpretable, robust if background and signal are well-separated.  
**Cons:** Sensitive to threshold choice; struggles when background and signal overlap.

#### 2. Gaussian Mixture Model (`assign_by_gmm`)
For each guide, fit a two-component Gaussian mixture model to the distribution of
log-transformed counts across all cells. Component 1 = background (low counts),
Component 2 = signal (high counts). A cell is assigned to a guide if the posterior
probability of belonging to the signal component exceeds 0.5.

**Pros:** Data-driven threshold; adapts to each guide's distribution.  
**Cons:** Assumes bimodal distribution; may fail if guide delivery efficiency is very low.

#### 3. Percentile threshold (`assign_percentile_threshold`)
Assign the top X% of cells (by count) for each guide as positive. This is guide-agnostic
and handles varying total counts across cells.

### Quality metrics after assignment
- **Cells per guide**: should be roughly equal across guides (consistent delivery)
- **Fraction with 0 guides assigned**: cells that received no guide RNA
- **Fraction with >1 guide**: multi-guide cells (may need to be excluded or analysed separately)
- **Guide efficiency**: fraction of cells that received any guide at all

In [ ]:
if adata_guide is None:
    print('Guide RNA modality not available — skipping guide assignment demo.')
else:
    print('Guide RNA matrix shape (cells x guides):', adata_guide.shape)

    guide_matrix = adata_guide.X.toarray() if sp.issparse(adata_guide.X) else np.array(adata_guide.X)
    print(f'Count range: [{guide_matrix.min():.0f}, {guide_matrix.max():.0f}]')
    print(f'Fraction of zero entries: {(guide_matrix == 0).mean():.1%}')

    # Summary per cell
    total_per_cell = guide_matrix.sum(axis=1)
    max_per_cell   = guide_matrix.max(axis=1)
    print(f'\nTotal guide UMIs per cell:  mean={total_per_cell.mean():.1f},  median={np.median(total_per_cell):.1f}')
    print(f'Max single guide per cell:  mean={max_per_cell.mean():.1f},  median={np.median(max_per_cell):.1f}')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(total_per_cell, bins=60, color='steelblue', edgecolor='none')
    axes[0].set_xlabel('Total guide UMIs per cell')
    axes[0].set_ylabel('Number of cells')
    axes[0].set_title('Total guide UMI distribution')

    axes[1].hist(max_per_cell, bins=60, color='steelblue', edgecolor='none')
    axes[1].axvline(5, color='red', ls='--', lw=1.5, label='threshold = 5 UMI')
    axes[1].set_xlabel('Max single-guide UMI count per cell')
    axes[1].set_ylabel('Number of cells')
    axes[1].set_title('Max guide count per cell (signal)')
    axes[1].legend()

    plt.suptitle('Guide RNA count distributions — papalexi_2021', y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / '00c_guide_distributions.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
if adata_guide is None:
    print('Guide RNA modality not available — skipping.')
else:
    # Threshold-based assignment.
    # assign_by_threshold marks each guide-cell entry as 1 (positive) or 0 (negative)
    # based on the raw UMI count compared to assignment_threshold.
    # The result is stored as a new layer in adata_guide.
    ga = pt.pp.GuideAssignment()
    ga.assign_by_threshold(
        adata_guide,
        assignment_threshold=5,
        output_layer='guide_binary',
    )

    # Retrieve binary assignment matrix
    if 'guide_binary' in adata_guide.layers:
        binary = adata_guide.layers['guide_binary']
        binary = binary.toarray() if sp.issparse(binary) else np.array(binary)
        n_guides_per_cell = binary.sum(axis=1)
        print('Guide assignment complete (threshold = 5 UMI).\n')
        print('Number of guides assigned per cell:')
        vals, cnts = np.unique(n_guides_per_cell, return_counts=True)
        for v, c in zip(vals, cnts):
            print(f'  {int(v):2d} guide(s) assigned: {c:5d} cells ({c / len(adata_guide):.1%})')
    else:
        # Some versions store results in obs instead
        print('Binary layer not found. Available layers:', list(adata_guide.layers.keys()))
        print('obs columns after assignment:', adata_guide.obs.columns.tolist())

In [ ]:
if adata_guide is not None and 'guide_binary' in adata_guide.layers:
    binary = adata_guide.layers['guide_binary']
    binary = binary.toarray() if sp.issparse(binary) else np.array(binary)

    # Cells per guide (how many cells received each guide)
    cells_per_guide = binary.sum(axis=0)
    guide_names     = adata_guide.var_names

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Histogram of cells per guide — should be roughly uniform if delivery was consistent
    axes[0].hist(cells_per_guide, bins=40, color='steelblue', edgecolor='none')
    axes[0].set_xlabel('Cells assigned to guide')
    axes[0].set_ylabel('Number of guides')
    axes[0].set_title('Cells per guide (delivery uniformity)')

    # Bar chart of top 20 most-assigned guides
    top_idx  = np.argsort(cells_per_guide)[::-1][:20]
    top_names = guide_names[top_idx]
    top_counts = cells_per_guide[top_idx]
    axes[1].barh(range(20), top_counts[::-1], color='steelblue')
    axes[1].set_yticks(range(20))
    axes[1].set_yticklabels(top_names[::-1], fontsize=8)
    axes[1].set_xlabel('Cells assigned')
    axes[1].set_title('Top 20 guides by cell count')

    plt.suptitle('Guide assignment summary — threshold method (>=5 UMI)', y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / '00c_guide_assignment.pdf', bbox_inches='tight')
    plt.show()
else:
    print('Skipping guide assignment visualization (data unavailable).')

<a id='mixscape'></a>
## 4. MIXSCAPE — classifying genuine KO cells from non-perturbed cells

### The fundamental problem with CRISPR screens

In CRISPRi and CRISPRa screens, guide RNA delivery does **not guarantee** successful
perturbation. A cell may:
1. **Receive the guide** but have the dCas9-effector fail to bind the target locus
2. **Have the target gene knocked down** but have it recover via compensatory mechanisms
3. **Receive the guide and be genuinely perturbed** (the outcome we want)

If you treat all cells in a perturbation group as equivalent, you are mixing two
subpopulations:
- **KO cells** (genuine knockdown) — show the expected transcriptional signature
- **NP cells** (non-perturbed) — look like control cells

This dilution biases DE results toward the null and reduces statistical power.

### The MIXSCAPE algorithm (Papalexi et al., Nature Genetics 2021)

**Step 1 — Perturbation signature (`perturbation_signature`)**

For each perturbed cell $i$, compute its **perturbation signature** $s_i$:

1. Find the $k$ nearest **control** cells to cell $i$ in PCA space (typically $k=20$)
2. Compute the mean expression of those $k$ control cells: $\bar{x}_{\text{ctrl}}^{(i)}$
3. Perturbation signature = $s_i = x_i - \bar{x}_{\text{ctrl}}^{(i)}$

**Why this works:** The perturbation signature subtracts away the "background"
cell state that the perturbed cell would have had without the perturbation.
For a genuinely perturbed cell, $s_i$ will reflect the perturbation effect.
For an NP cell, $s_i \approx 0$ (it looks like its local control neighbours).

**Step 2 — Gaussian mixture model classification (`mixscape`)**

For each perturbation group $p$ with $n_p$ cells:

1. Project all perturbation signatures $\{s_1, \ldots, s_{n_p}\}$ onto the **first
   principal component** of those signatures within that group.
2. Fit a **2-component Gaussian mixture model (GMM)** to the 1-D projection:
   - Component 1 (mean ≈ 0): the **NP subpopulation** (non-perturbed)
   - Component 2 (mean shifted): the **KO subpopulation** (genuine knockdown)
3. Assign each cell to the component with higher posterior probability.
4. The mixture weight of the KO component estimates **perturbation efficiency**.

**Why 1-D GMM?**
Fitting a GMM in 50-dimensional PCA space is statistically expensive and prone to
local minima. By first projecting to the single direction that maximally separates
KO from NP cells (the first PC of the perturbation signatures), we reduce the problem
to a tractable 1-D mixture model while capturing the most relevant variation.

**Step 3 — LDA visualisation (`lda`)**
Linear Discriminant Analysis (LDA) is then applied to the KO cells across all
perturbation groups to produce a low-dimensional embedding that maximises separation
between different perturbation programs. This is useful for:
- Identifying perturbations with similar downstream effects
- Validating that KO cells separate by perturbation identity (QC check)
- Visualising the structure of the screen

In [ ]:
# Preprocess RNA modality for Mixscape.
# Mixscape operates on normalised log-expression (not PCA directly),
# but we also compute PCA to use as the representation for the k-NN search
# in the perturbation_signature step.
adata_rna_mxs = adata_rna.copy()

sc.pp.normalize_total(adata_rna_mxs, target_sum=1e4)
sc.pp.log1p(adata_rna_mxs)
sc.pp.highly_variable_genes(adata_rna_mxs, n_top_genes=2000, subset=False)
sc.pp.scale(adata_rna_mxs, max_value=10)

# Fix any NaN introduced by zero-variance genes after scaling
X_arr = adata_rna_mxs.X.toarray() if sp.issparse(adata_rna_mxs.X) else adata_rna_mxs.X
if np.isnan(X_arr).any():
    X_arr = np.nan_to_num(X_arr, nan=0.0)
    adata_rna_mxs.X = sp.csr_matrix(X_arr)

sc.tl.pca(adata_rna_mxs, n_comps=40)
sc.pp.neighbors(adata_rna_mxs, n_neighbors=20)

print('Preprocessed RNA for Mixscape:', adata_rna_mxs.shape)
print('PCA computed:', 'X_pca' in adata_rna_mxs.obsm)
print('Control cells (NT):', (adata_rna_mxs.obs['perturbation'] == 'NT').sum())

In [ ]:
mxs = pt.tl.Mixscape()

# Step 1: perturbation_signature
# For every non-control cell, find its k nearest CONTROL neighbours in PCA space
# and subtract their mean expression. The result (stored in layers['X_pert']) is
# the per-cell perturbation signature: approximately zero for NP cells,
# positive/negative for genes genuinely changed by the perturbation.
#
# pert_key   : obs column with perturbation labels
# control_key: the value in pert_key that identifies non-targeting control cells
# n_neighbors: k for the kNN search (default 20; more neighbours = smoother signature)
# use_rep    : embedding for the kNN search — PCA is standard
mxs.perturbation_signature(
    adata_rna_mxs,
    pert_key='perturbation',
    control_key='NT',
    n_neighbors=20,
    use_rep='X_pca',
)

print('Perturbation signature computed.')
print('Layers now available:', list(adata_rna_mxs.layers.keys()))

# Quick sanity check: NP cells should have signature values near 0
# while strongly perturbed cells should show larger absolute values
if 'X_pert' in adata_rna_mxs.layers:
    pert_layer = adata_rna_mxs.layers['X_pert']
    pert_layer = pert_layer.toarray() if sp.issparse(pert_layer) else np.array(pert_layer)

    nt_mask = adata_rna_mxs.obs['perturbation'] == 'NT'
    sig_magnitude = np.abs(pert_layer).mean(axis=1)   # mean absolute signature per cell

    print(f'\nMean |perturbation signature| per cell:')
    print(f'  NT (control) cells: {sig_magnitude[nt_mask].mean():.4f}')
    print(f'  Perturbed cells:    {sig_magnitude[~nt_mask].mean():.4f}')
    print('  (Perturbed cells should have larger signatures if perturbation worked.)')

In [ ]:
# Step 2: Mixscape classification
# For each perturbation group, fit a 2-component GMM to the 1-D projection
# of perturbation signatures. Assigns each cell to 'KO' or 'NP'.
#
# control      : the label for non-targeting control cells in labels_key
# labels_key   : obs column used to group cells by perturbation identity
# layer        : which layer contains the perturbation signatures (from step 1)
# pval_cutoff  : threshold for rejecting the null (NP component dominates);
#                perturbations where all cells look like controls get flagged
pert_layer_key = 'X_pert' if 'X_pert' in adata_rna_mxs.layers else None

if pert_layer_key is not None:
    mxs.mixscape(
        adata_rna_mxs,
        control='NT',
        labels_key='perturbation',
        layer=pert_layer_key,
        pval_cutoff=0.05,
    )

    # Find the obs column(s) added by Mixscape
    mix_cols = [c for c in adata_rna_mxs.obs.columns if 'mixscape' in c.lower()]
    print('Mixscape-added obs columns:', mix_cols)
    print()

    if mix_cols:
        global_col = [c for c in mix_cols if 'global' in c]
        show_col   = global_col[0] if global_col else mix_cols[0]
        print(f'Classification summary ({show_col}):')
        print(adata_rna_mxs.obs[show_col].value_counts())
        print()
        print('Per-perturbation breakdown (sample):')
        print(adata_rna_mxs.obs[['perturbation', show_col]]
              .value_counts().head(20).to_string())
else:
    print('Skipping Mixscape classification: X_pert layer not found.')

In [ ]:
mix_cols = [c for c in adata_rna_mxs.obs.columns if 'mixscape' in c.lower()]
global_col = ([c for c in mix_cols if 'global' in c] or mix_cols or [None])[0]

if global_col and 'X_pert' in adata_rna_mxs.layers:
    pert_layer = adata_rna_mxs.layers['X_pert']
    pert_layer = pert_layer.toarray() if sp.issparse(pert_layer) else np.array(pert_layer)

    # Plot 1: Distribution of mean |perturbation signature| for KO vs NP vs NT cells
    sig_mag = np.abs(pert_layer).mean(axis=1)
    adata_rna_mxs.obs['sig_magnitude'] = sig_mag

    classes     = adata_rna_mxs.obs[global_col].unique()
    palette     = {'KO': 'tomato', 'NP': 'steelblue', 'NT': 'lightgray'}
    class_order = [c for c in ['KO', 'NP', 'NT'] if c in classes]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Violin: perturbation signature magnitude by Mixscape class
    class_data = {c: sig_mag[adata_rna_mxs.obs[global_col] == c] for c in class_order}
    positions  = range(len(class_order))
    vp = axes[0].violinplot(list(class_data.values()), positions=list(positions),
                            showmedians=True)
    for pc, cls in zip(vp['bodies'], class_order):
        pc.set_facecolor(palette.get(cls, 'gray'))
        pc.set_alpha(0.7)
    axes[0].set_xticks(list(positions))
    axes[0].set_xticklabels(class_order)
    axes[0].set_ylabel('Mean |perturbation signature|')
    axes[0].set_title('Mixscape classification\nvs. perturbation signature magnitude')

    # Bar: KO efficiency per perturbation (fraction classified as KO)
    if global_col in adata_rna_mxs.obs.columns:
        eff = (adata_rna_mxs.obs
               .query('perturbation != "NT"')
               .groupby('perturbation')[global_col]
               .apply(lambda x: (x == 'KO').mean())
               .sort_values(ascending=False))
        top_eff = eff.head(20)
        axes[1].bar(range(len(top_eff)), top_eff.values, color='steelblue')
        axes[1].set_xticks(range(len(top_eff)))
        axes[1].set_xticklabels(top_eff.index, rotation=45, ha='right', fontsize=8)
        axes[1].set_ylabel('Fraction of cells classified as KO')
        axes[1].set_title('Perturbation efficiency (top 20 perturbations)')
        axes[1].axhline(0.5, color='red', ls='--', lw=1, label='50% threshold')
        axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(FIG_DIR / '00c_mixscape_classification.pdf', bbox_inches='tight')
    plt.show()
else:
    print('Skipping Mixscape visualization: classification not available.')

<a id='cinema-ot'></a>
## 5. CINEMA-OT — separating causal perturbation effects from confounding cell state

### The confounder problem in Perturb-seq

When you compare a perturbed cell to a control cell, any gene expression difference
could arise from two fundamentally different sources:

1. **Intrinsic effect**: the genuine causal effect of the perturbation (what we want)
2. **Extrinsic effect**: confounding variation because the cells were already in
   different transcriptional states *before* the perturbation was applied

For example: suppose perturbation X activates an erythroid differentiation program.
If control cells happen to contain more pre-erythroid cells (by chance, or due to
uneven guide delivery during cell cycle phases), the DE analysis will
*over-estimate* the perturbation effect because it conflates the genuine activation
with pre-existing cell-state differences.

### The CINEMA-OT approach (Dong et al., Nature Methods 2023)

CINEMA-OT (Causal INtEraction via Matching Alignments for Optimal Transport) decomposes
the observed transcriptional shift into:

$$\underbrace{x_{\text{observed}}^{\text{pert}}}_{\text{perturbed cell}} = \underbrace{x_{\text{intrinsic}}}_{\text{genuine effect}} + \underbrace{x_{\text{extrinsic}}}_{\text{cell-state confound}}$$

**Step 1 — Confounder test (HSIC)**

The Hilbert-Schmidt Independence Criterion (HSIC) tests whether the perturbation
assignment is statistically independent of a confounding variable (e.g., cell cycle
phase or pre-existing cell state). If HSIC is large (p < threshold), the perturbation
group and control group have non-overlapping cell state distributions and a simple
cell matching step is needed.

**Step 2 — Optimal transport matching**

For each perturbed cell $x_i^{\text{pert}}$, find the control cell (or weighted
combination of control cells) that best represents the counterfactual state — *what
would this cell look like without the perturbation?*

This is formulated as an **optimal transport** (OT) problem: find a transport plan
$T$ (a matrix of non-negative weights) such that:
- Row $i$ of $T$: weights over control cells for matching to perturbed cell $i$
- The total transport cost (sum of distances × weights) is minimised
- Row sums = 1 (each perturbed cell's mass is fully transported to control cells)

The **Sinkhorn algorithm** (entropic regularisation of OT) is used for computational
efficiency. The regularisation parameter $\lambda$ controls the smoothness:
- Large $\lambda$ → diffuse matching (each perturbed cell is matched to many controls)
- Small $\lambda$ → sharp matching (each perturbed cell is matched to few controls)

**Step 3 — Intrinsic/extrinsic decomposition**

Given the transport plan $T$, the matched control expression for perturbed cell $i$ is:

$$\hat{x}_{\text{ctrl}}^{(i)} = \sum_j T_{ij} \cdot x_j^{\text{ctrl}}$$

Then:
- **Extrinsic** (confounding cell state): $\hat{x}_{\text{ctrl}}^{(i)}$ (the matched control)
- **Intrinsic** (causal perturbation effect): $x_i^{\text{pert}} - \hat{x}_{\text{ctrl}}^{(i)}$

**Why this is better than simply subtracting a global control mean:**
The global mean ignores cell-state heterogeneity. If a perturbed cell happens to be
in a high-proliferation state, subtracting the global control mean will give a residual
that partly reflects proliferation, not the perturbation. CINEMA-OT finds the specific
control cells in the same cell state, giving a cleaner estimate of the causal effect.

In [ ]:
# Use CEBPE from Norman 2019 (single perturbations only) as a demonstration.
# CEBPE is one of the strongest-effect perturbations in this dataset.
# We subsample to keep CINEMA-OT computationally tractable for this tutorial.
TARGET_CT = 'CEBPE'

adata_ct_full = adata[
    adata.obs['perturbation'].isin([TARGET_CT, 'control']) &
    (adata.obs['nperts'] == 1)
].copy()

# Subsample: 400 cells each of perturbed and control
rng_ct   = np.random.default_rng(7)
ctrl_idx = np.where(adata_ct_full.obs['perturbation'] == 'control')[0]
pert_idx = np.where(adata_ct_full.obs['perturbation'] == TARGET_CT)[0]
ctrl_idx = rng_ct.choice(ctrl_idx, size=min(400, len(ctrl_idx)), replace=False)
pert_idx = rng_ct.choice(pert_idx, size=min(400, len(pert_idx)), replace=False)

adata_ct = adata_ct_full[np.concatenate([ctrl_idx, pert_idx])].copy()

# Preprocess
sc.pp.normalize_total(adata_ct, target_sum=1e4)
sc.pp.log1p(adata_ct)
sc.pp.highly_variable_genes(adata_ct, n_top_genes=2000, subset=True)
sc.tl.pca(adata_ct, n_comps=20)

n_p = (adata_ct.obs['perturbation'] == TARGET_CT).sum()
n_c = (adata_ct.obs['perturbation'] == 'control').sum()
print(f'CINEMA-OT subset: {adata_ct.shape}')
print(f'  {TARGET_CT} cells: {n_p}')
print(f'  control cells:   {n_c}')

In [ ]:
ct = pt.tl.CinemaOt()

# causaleffect() performs the full CINEMA-OT pipeline:
#   1. HSIC confounder test
#   2. Sinkhorn optimal transport matching
#   3. Intrinsic / extrinsic decomposition
#
# Parameters:
#   pert_key     : obs column with perturbation labels
#   control      : value in pert_key identifying control cells
#   return_matching: if True, returns the transport plan matrix
#   thres        : HSIC p-value threshold for declaring confounding present.
#                  If the test p > thres, CINEMA-OT skips the OT step
#                  (no confounding detected) and uses identity matching.
#   smoothness   : entropic regularisation lambda for Sinkhorn OT.
#                  Smaller = sharper (more 1-1) matching;
#                  Larger  = smoother (each cell distributed over more partners).
#   eps          : Sinkhorn convergence tolerance
#   solver       : 'Sinkhorn' (fast, approximate) or 'exact' (slower, exact OT)
try:
    transport_plan = ct.causaleffect(
        adata_ct,
        pert_key='perturbation',
        control='control',
        return_matching=True,
        thres=0.5,
        smoothness=1e-5,
        eps=1e-3,
        solver='Sinkhorn',
    )
    print('CINEMA-OT complete.')
    if transport_plan is not None and hasattr(transport_plan, 'shape'):
        print(f'Transport plan shape: {transport_plan.shape}  (perturbed x control cells)')
    # Check what was stored in adata
    new_obsm = [k for k in adata_ct.obsm if 'cinema' in k.lower() or 'causal' in k.lower()]
    new_uns  = [k for k in adata_ct.uns  if 'cinema' in k.lower() or 'causal' in k.lower()]
    print(f'New obsm keys: {new_obsm}')
    print(f'New uns keys:  {new_uns}')
except Exception as e:
    print(f'CINEMA-OT raised: {e}')
    transport_plan = None
    print('This can happen if optional dependencies (ot, pot) are missing.')
    print('Install with: pip install POT')

In [ ]:
if transport_plan is not None and hasattr(transport_plan, 'shape'):
    T = np.array(transport_plan)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # --- Plot 1: Transport plan heatmap ---
    # T[i, j] = mass transported from perturbed cell i to control cell j.
    # Bright squares indicate strong (confident) matches.
    # An identity-like diagonal would mean 1-to-1 matching;
    # a uniform matrix would mean each perturbed cell is matched to all controls equally.
    n_show = min(80, T.shape[0], T.shape[1])
    im = axes[0].imshow(T[:n_show, :n_show], aspect='auto', cmap='viridis')
    axes[0].set_xlabel('Control cell index')
    axes[0].set_ylabel('Perturbed cell index')
    axes[0].set_title(f'OT transport plan\n(first {n_show} cells each)')
    plt.colorbar(im, ax=axes[0], label='Transport weight')

    # --- Plot 2: Row entropy of transport plan ---
    # Entropy of row i = how diffusely perturbed cell i is matched.
    # Low entropy = sharp matching to few controls (confident match)
    # High entropy = diffuse matching to many controls (uncertain match)
    row_entropy = -(T * np.log(T + 1e-10)).sum(axis=1)
    axes[1].hist(row_entropy, bins=40, color='steelblue', edgecolor='none')
    axes[1].set_xlabel('Row entropy of transport plan')
    axes[1].set_ylabel('Number of perturbed cells')
    axes[1].set_title('Matching confidence\n(lower entropy = sharper matching)')

    # --- Plot 3: Intrinsic effect magnitude vs. total expression ---
    # Intrinsic effect = perturbed cell expression - matched control expression
    pert_mask = adata_ct.obs['perturbation'] == TARGET_CT
    X_pert    = adata_ct[pert_mask].X
    X_ctrl    = adata_ct[~pert_mask].X
    X_pert    = X_pert.toarray()  if sp.issparse(X_pert)  else np.array(X_pert)
    X_ctrl    = X_ctrl.toarray()  if sp.issparse(X_ctrl)  else np.array(X_ctrl)

    # Matched control expression for each perturbed cell
    X_ctrl_matched = T @ X_ctrl                    # (n_pert, n_genes)
    intrinsic      = X_pert - X_ctrl_matched       # (n_pert, n_genes)

    intrinsic_mag = np.abs(intrinsic).mean(axis=1)
    total_expr    = X_pert.mean(axis=1)

    axes[2].scatter(total_expr, intrinsic_mag, s=15, alpha=0.5, color='steelblue')
    axes[2].set_xlabel('Mean expression (perturbed cell)')
    axes[2].set_ylabel('Mean |intrinsic effect|')
    axes[2].set_title(f'Causal perturbation effect magnitude\n{TARGET_CT} vs. control')

    plt.suptitle('CINEMA-OT results — optimal transport decomposition', y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'00c_cinemaot_{TARGET_CT}.pdf', bbox_inches='tight')
    plt.show()

    # Top genes with largest intrinsic effect (causally regulated by CEBPE)
    intrinsic_mean = intrinsic.mean(axis=0)      # mean per gene
    top_up   = np.argsort(intrinsic_mean)[::-1][:10]
    top_down = np.argsort(intrinsic_mean)[:10]
    print('Top 10 genes with strongest INTRINSIC UP-regulation by CEBPE:')
    for i in top_up:
        print(f'  {adata_ct.var_names[i]:12s}  mean intrinsic effect = {intrinsic_mean[i]:.3f}')
    print()
    print('Top 10 genes with strongest INTRINSIC DOWN-regulation by CEBPE:')
    for i in top_down:
        print(f'  {adata_ct.var_names[i]:12s}  mean intrinsic effect = {intrinsic_mean[i]:.3f}')
else:
    print('Transport plan not available — skipping CINEMA-OT visualization.')

In [ ]:
print('=' * 60)
print('PHASE 1 pertpy QUICKSTART — SUMMARY CHECKLIST')
print('=' * 60)
print()

checks = {
    'pt.data.norman_2019() loaded successfully'   : adata is not None,
    'pt.data.papalexi_2021() loaded successfully'  : adata_rna is not None,
    'Guide RNA modality available'                 : adata_guide is not None,
    'Differential response (Wilcoxon) run'         : 'rank_genes_wilcoxon' in adata_de.uns,
    'DE volcano plot produced'                     : True,
    'Guide assignment run'                         : 'guide_binary' in (adata_guide.layers if adata_guide is not None else {}),
    'Mixscape perturbation signature computed'     : 'X_pert' in adata_rna_mxs.layers,
    'Mixscape classification run'                  : any('mixscape' in c for c in adata_rna_mxs.obs.columns),
    'CINEMA-OT causal effect computed'             : transport_plan is not None,
}

all_pass = True
for check, result in checks.items():
    status = 'v' if result else 'X'
    print(f'  [{status}]  {check}')
    if not result:
        all_pass = False

print()
print('Phase 1 pertpy quickstart:', 'PASSED' if all_pass else 'PARTIAL (see above)')
print()
print('Key pertpy classes used:')
print('  pt.data.*           — built-in datasets')
print('  sc.tl.rank_genes_groups — Wilcoxon DE (via scanpy)')
print('  pt.pp.GuideAssignment  — guide RNA assignment')
print('  pt.tl.Mixscape         — CRISPR perturbation classification')
print('  pt.tl.CinemaOt         — causal OT decomposition')

In [ ]:
from datetime import datetime
import subprocess

timestamp  = datetime.now().strftime('%Y%m%d_%H%M')
report_dir = Path('../../results/reports')
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / f'00c_pertpy_quickstart_{timestamp}.html'

nb_path = (
    __vsc_ipynb_file__
    if '__vsc_ipynb_file__' in dir()
    else '00c_pertpy_quickstart.ipynb'
)

subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'html', '--output', str(report_path), nb_path],
    check=False,
)
print(f'Report saved to {report_path}')